# 08 — Crossover Operators (PMX, OX, CX)

**Maps to:** `report/Chapters/Task2.tex` §`T2:Crossover`.  
**Ticket:** TICKET-08 (risk point: allocate full week).

Implement PMX, OX, CX. For each, show:
- worked example on a toy case with a fixed cut/segment;
- whether the operator can produce infeasible offspring (feeds TICKET-11 in `notebooks/11_infeasibility_analysis.ipynb`).

In [1]:
import numpy as np

SEED = 12
rng = np.random.default_rng(SEED)

## Test Parents

All worked examples use the same 8-city parent pair and cut points `[2, 4]`
so the three operators can be compared directly.

In [2]:
pa = np.array([0, 1, 2, 3, 4, 5, 6, 7])
pb = np.array([2, 6, 4, 0, 5, 7, 1, 3])
PT1, PT2 = 2, 4

print(f"Parent A: {pa}")
print(f"Parent B: {pb}")
print(f"Cut points: [{PT1}, {PT2}]  →  segment: {pa[PT1:PT2+1]}")

Parent A: [0 1 2 3 4 5 6 7]
Parent B: [2 6 4 0 5 7 1 3]
Cut points: [2, 4]  →  segment: [2 3 4]


## Helper — Feasibility Check

In [3]:
def is_valid_permutation(chromosome, n_cities):
    """Check if chromosome is a valid permutation of 0..n_cities-1."""
    return set(chromosome) == set(range(n_cities))

---
## 1. Partially-Mapped Crossover (PMX)

PMX preserves the **absolute positions** of genes in the crossover segment
from Parent A, and resolves displaced genes from Parent B through a mapping chain.

### Algorithm

1. Select two crossover points.
2. Copy segment from Parent A into child.
3. For each value in Parent B's segment that is NOT already in the child,
   follow the mapping chain to find an empty position.
4. Fill remaining empty positions directly from Parent B.

### PMX — Step-by-Step Diagram

```
Parent A: [ 0  1 | 2  3  4 | 5  6  7 ]
Parent B: [ 2  6 | 4  0  5 | 7  1  3 ]
                   pt1=2     pt2=4

Step 1 — Copy segment from Parent A:

  Child:  [ _  _ | 2  3  4 | _  _  _ ]
  Segment values: {2, 3, 4}

Step 2 — Map Parent B's segment values not yet in child:

  B[2]=4  → already in child, skip

  B[3]=0  → not in child, follow mapping chain:
           A[3]=3 → find 3 in B → B[7]
           position 7 is outside segment → place 0 there
  Child:  [ _  _   2  3  4   _  _  0 ]

  B[4]=5  → not in child, follow mapping chain:
           A[4]=4 → find 4 in B → B[2]
           position 2 is inside segment → keep following:
           A[2]=2 → find 2 in B → B[0]
           position 0 is outside segment → place 5 there
  Child:  [ 5  _   2  3  4   _  _  0 ]

Step 3 — Fill remaining from Parent B:

  Child[1] = B[1] = 6
  Child[5] = B[5] = 7
  Child[6] = B[6] = 1

  Child:  [ 5  6   2  3  4   7  1  0 ]
           ✓ valid permutation of {0..7}
```

### PMX — Implementation

In [4]:
def pmx(parent_a, parent_b, pt1, pt2):
    n = len(parent_a)
    child = np.full(n, -1, dtype=parent_a.dtype)

    # Step 1: copy segment from parent_a
    child[pt1:pt2 + 1] = parent_a[pt1:pt2 + 1]
    segment_vals = set(child[pt1:pt2 + 1])

    # Step 2: map values from parent_b's segment that aren't placed yet
    for i in range(pt1, pt2 + 1):
        val = parent_b[i]
        if val not in segment_vals:
            j = i
            while pt1 <= j <= pt2:
                j = np.where(parent_b == parent_a[j])[0][0]
            child[j] = val

    # Step 3: fill remaining from parent_b
    empty = child == -1
    child[empty] = parent_b[empty]

    return child

### PMX — Worked Example

In [5]:
child_pmx = pmx(pa, pb, PT1, PT2)

print(f"Parent A:  {pa}")
print(f"Parent B:  {pb}")
print(f"Segment:   positions [{PT1}, {PT2}] → {pa[PT1:PT2+1]}")
print(f"Child PMX: {child_pmx}")
print(f"Valid:     {is_valid_permutation(child_pmx, len(pa))}")

Parent A:  [0 1 2 3 4 5 6 7]
Parent B:  [2 6 4 0 5 7 1 3]
Segment:   positions [2, 4] → [2 3 4]
Child PMX: [5 6 2 3 4 7 1 0]
Valid:     True


### PMX — Unit Test (1000 random crossovers)

In [6]:
rng_test = np.random.default_rng(SEED)
n = 20

for trial in range(1000):
    a = rng_test.permutation(n)
    b = rng_test.permutation(n)
    pts = sorted(rng_test.choice(n, size=2, replace=False))
    child = pmx(a, b, pts[0], pts[1])
    assert is_valid_permutation(child, n), f"PMX invalid on trial {trial}: {child}"

print("PMX: 1000/1000 trials produced valid permutations.")

PMX: 1000/1000 trials produced valid permutations.


---
## 2. Order Crossover (OX)

OX preserves the **relative order** of genes from Parent B while keeping
a contiguous segment from Parent A.

### Algorithm

1. Select two crossover points.
2. Copy segment from Parent A into child.
3. Starting from `pt2+1` (wrapping around), collect values from Parent B
   in order, skipping any already in the child.
4. Fill empty positions in the child from `pt2+1` onwards with the collected values.

### OX — Step-by-Step Diagram

```
Parent A: [ 0  1 | 2  3  4 | 5  6  7 ]
Parent B: [ 2  6 | 4  0  5 | 7  1  3 ]
                   pt1=2     pt2=4

Step 1 — Copy segment from Parent A:

  Child:  [ _  _ | 2  3  4 | _  _  _ ]
  Segment values: {2, 3, 4}

Step 2 — Scan Parent B starting at pt2+1=5, wrapping around.
         Collect values NOT in {2, 3, 4}:

  B[5]=7 ✓  B[6]=1 ✓  B[7]=3 ✗  B[0]=2 ✗
  B[1]=6 ✓  B[2]=4 ✗  B[3]=0 ✓  B[4]=5 ✓

  Remaining (in order): [7, 1, 6, 0, 5]

Step 3 — Fill empty positions starting at pt2+1=5, wrapping:

  pos 5 ← 7    pos 6 ← 1    pos 7 ← 6
  pos 0 ← 0    pos 1 ← 5

  Child:  [ 0  5   2  3  4   7  1  6 ]
           ✓ valid permutation of {0..7}
```

### OX — Implementation

In [7]:
def ox(parent_a, parent_b, pt1, pt2):
    n = len(parent_a)
    child = np.full(n, -1, dtype=parent_a.dtype)

    # Step 1: copy segment from parent_a
    child[pt1:pt2 + 1] = parent_a[pt1:pt2 + 1]
    segment_vals = set(child[pt1:pt2 + 1])

    # Step 2: collect remaining values from parent_b in order, starting at pt2+1
    remaining = []
    for i in range(n):
        idx = (pt2 + 1 + i) % n
        if parent_b[idx] not in segment_vals:
            remaining.append(parent_b[idx])

    # Step 3: fill empty positions starting at pt2+1
    j = 0
    for i in range(n):
        idx = (pt2 + 1 + i) % n
        if child[idx] == -1:
            child[idx] = remaining[j]
            j += 1

    return child

### OX — Worked Example

In [8]:
child_ox = ox(pa, pb, PT1, PT2)

print(f"Parent A:  {pa}")
print(f"Parent B:  {pb}")
print(f"Segment:   positions [{PT1}, {PT2}] → {pa[PT1:PT2+1]}")
print(f"Child OX:  {child_ox}")
print(f"Valid:     {is_valid_permutation(child_ox, len(pa))}")

Parent A:  [0 1 2 3 4 5 6 7]
Parent B:  [2 6 4 0 5 7 1 3]
Segment:   positions [2, 4] → [2 3 4]
Child OX:  [0 5 2 3 4 7 1 6]
Valid:     True


### OX — Unit Test (1000 random crossovers)

In [9]:
rng_test = np.random.default_rng(SEED)
n = 20

for trial in range(1000):
    a = rng_test.permutation(n)
    b = rng_test.permutation(n)
    pts = sorted(rng_test.choice(n, size=2, replace=False))
    child = ox(a, b, pts[0], pts[1])
    assert is_valid_permutation(child, n), f"OX invalid on trial {trial}: {child}"

print("OX: 1000/1000 trials produced valid permutations.")

OX: 1000/1000 trials produced valid permutations.


---
## 3. Cycle Crossover (CX)

CX identifies **cycles** in the position-wise mapping between parents.
Each gene in the child comes from one parent at the **same position**, so
feasibility is guaranteed by construction — no repair needed.

### Algorithm

1. Start at position 0. Trace the cycle:
   `parent_b[i]` → find its position in `parent_a` → repeat until returning to start.
2. Even-numbered cycles copy from Parent A, odd-numbered from Parent B.
3. Repeat for all unvisited positions.

### CX — Step-by-Step Diagram

```
           pos:  0  1  2  3  4  5  6  7
Parent A:      [ 0  1  2  3  4  5  6  7 ]
Parent B:      [ 2  6  4  0  5  7  1  3 ]

Cycle 1 (start at pos 0):

  pos 0: B[0]=2  → find 2 in A → pos 2
  pos 2: B[2]=4  → find 4 in A → pos 4
  pos 4: B[4]=5  → find 5 in A → pos 5
  pos 5: B[5]=7  → find 7 in A → pos 7
  pos 7: B[7]=3  → find 3 in A → pos 3
  pos 3: B[3]=0  → find 0 in A → pos 0  ← back to start!

  Cycle 1 positions: {0, 2, 3, 4, 5, 7}
  Even cycle → take from Parent A

Cycle 2 (start at pos 1 — first unvisited):

  pos 1: B[1]=6  → find 6 in A → pos 6
  pos 6: B[6]=1  → find 1 in A → pos 1  ← back to start!

  Cycle 2 positions: {1, 6}
  Odd cycle → take from Parent B

Assemble child:

  pos:    0  1  2  3  4  5  6  7
  source: A  B  A  A  A  A  B  A
  Child: [0  6  2  3  4  5  1  7]
          ✓ valid permutation of {0..7}
```

### CX — Implementation

In [10]:
def cx(parent_a, parent_b):
    n = len(parent_a)
    child = np.full(n, -1, dtype=parent_a.dtype)
    visited = np.zeros(n, dtype=bool)

    pos_in_a = {val: idx for idx, val in enumerate(parent_a)}

    cycle_num = 0
    for start in range(n):
        if visited[start]:
            continue

        # Trace cycle
        cycle_indices = []
        i = start
        while not visited[i]:
            visited[i] = True
            cycle_indices.append(i)
            i = pos_in_a[parent_b[i]]

        # Alternate source per cycle
        source = parent_a if cycle_num % 2 == 0 else parent_b
        for idx in cycle_indices:
            child[idx] = source[idx]

        cycle_num += 1

    return child, cycle_num

### CX — Worked Example

In [11]:
child_cx, n_cycles = cx(pa, pb)

print(f"Parent A:  {pa}")
print(f"Parent B:  {pb}")
print(f"Cycles:    {n_cycles}")
print(f"Child CX:  {child_cx}")
print(f"Valid:     {is_valid_permutation(child_cx, len(pa))}")

Parent A:  [0 1 2 3 4 5 6 7]
Parent B:  [2 6 4 0 5 7 1 3]
Cycles:    2
Child CX:  [0 6 2 3 4 5 1 7]
Valid:     True


### CX — Unit Test (1000 random crossovers)

In [12]:
rng_test = np.random.default_rng(SEED)
n = 20

for trial in range(1000):
    a = rng_test.permutation(n)
    b = rng_test.permutation(n)
    child, _ = cx(a, b)
    assert is_valid_permutation(child, n), f"CX invalid on trial {trial}: {child}"

print("CX: 1000/1000 trials produced valid permutations.")

CX: 1000/1000 trials produced valid permutations.


---
## 4. Feasibility Analysis

A key question for the repair mechanism (TICKET-12): **do these crossover
operators produce infeasible offspring?**

We compare the three permutation-aware operators above with a naive two-point
crossover that simply splices segments without any mapping.

### Naive Two-Point Crossover

```
Parent A: [ 0  1 | 2  3  4 | 5  6  7 ]
Parent B: [ 2  6 | 4  0  5 | 7  1  3 ]
                   pt1=2     pt2=4

Blindly splice A's segment into a copy of B:

  Child:  [ 2  6   2  3  4   7  1  3 ]
                   ↑ from A ↑

  Duplicates: {2, 3}    Missing: {0, 5}
  ✗ INVALID — not a permutation
```

In [13]:
def naive_two_point_crossover(parent_a, parent_b, pt1, pt2):
    """Standard two-point crossover — NOT permutation-aware."""
    child = parent_b.copy()
    child[pt1:pt2 + 1] = parent_a[pt1:pt2 + 1]
    return child

In [14]:
child_naive = naive_two_point_crossover(pa, pb, PT1, PT2)
dups = [x for x in child_naive if list(child_naive).count(x) > 1]
missing = sorted(set(range(len(pa))) - set(child_naive))

print(f"Child Naive: {child_naive}")
print(f"Duplicates:  {sorted(set(dups))}")
print(f"Missing:     {missing}")
print(f"Valid:       {is_valid_permutation(child_naive, len(pa))}")

Child Naive: [2 6 2 3 4 7 1 3]
Duplicates:  [np.int64(2), np.int64(3)]
Missing:     [0, 5]
Valid:       False


### Statistical Comparison (1000 trials, n=20)

In [15]:
rng_test = np.random.default_rng(SEED)
n = 20
n_trials = 1000
naive_invalid = 0

for _ in range(n_trials):
    a = rng_test.permutation(n)
    b = rng_test.permutation(n)
    pts = sorted(rng_test.choice(n, size=2, replace=False))
    child = naive_two_point_crossover(a, b, pts[0], pts[1])
    if not is_valid_permutation(child, n):
        naive_invalid += 1

print(f"Naive two-point: {naive_invalid}/{n_trials} trials produced INVALID permutations ({naive_invalid/n_trials*100:.1f}%)")
print()
print("Summary:")
print(f"  PMX   — always feasible (mapping chain resolves conflicts)")
print(f"  OX    — always feasible (relative-order fill skips duplicates)")
print(f"  CX    — always feasible by construction (each position from exactly one parent)")
print(f"  Naive — {naive_invalid/n_trials*100:.1f}% infeasible (duplicates + missing cities)")

Naive two-point: 997/1000 trials produced INVALID permutations (99.7%)

Summary:
  PMX   — always feasible (mapping chain resolves conflicts)
  OX    — always feasible (relative-order fill skips duplicates)
  CX    — always feasible by construction (each position from exactly one parent)
  Naive — 99.7% infeasible (duplicates + missing cities)


## Summary

| Operator | Preserves | Feasibility | Why |
|---|---|---|---|
| **PMX** | Absolute positions from segment | Always feasible | Mapping chain ensures every displaced value finds a unique, unoccupied position. |
| **OX** | Relative order from Parent B | Always feasible | Ordered fill from Parent B skips values already present in the copied segment. |
| **CX** | Exact positions (from one parent) | Always feasible | Each position takes its value from exactly one parent — no duplicates possible. |
| **Naive two-point** | Nothing | Usually infeasible | Blind segment swap creates duplicates and missing cities. |

**Implication for TICKET-12 (repair mechanism):**  
When using PMX, OX, or CX correctly, crossover alone does not produce infeasible
offspring. However, a repair mechanism is still valuable because:
1. Simpler/faster crossover operators (e.g., naive two-point) **do** produce infeasibles.
2. A repair fallback makes the GA robust against implementation bugs or operator variations.
3. The project spec requires comparing repair vs. alternative constraint-handling strategies
   (TICKET-13, TICKET-17), so infeasible solutions must be generated to exercise repair.